# Chapter 8: From molecule to circuit — the whole pipeline

Companion notebook to *Quantum Algorithms from First Principles*, Chapter 8,
"Building the whole pipeline" and "Testing the pipeline, and the sign trap", on the
canonical example: H$_2$ in STO-3G at 0.735 Å.

The six steps of the guide: (1) integrals from a classical package, checked against its
own Hartree–Fock energy; (2) the Bravyi–Kitaev matrix $\beta$ and its inverse over
$\mathbb{F}_2$; (3) the sets $U(j)$, $P(j)$ (row $j-1$!), $F(j)$, $R(j)$; (4) operators in
the symplectic representation (Qiskit's `SparsePauliOp` does the symplectic products);
(5) collect terms and drop those below $10^{-8}$; (6) circuits through `pauli_evolution`.
Then the five tests, in order.

**Conventions, fixed in one place.** $|1\rangle$ = occupied. $A^+ = (X-iY)/2$ raises.
Spin-orbitals are interleaved, $p = 2i+\sigma$ ($\sigma=0$ up, $1$ down), so the
Hartree–Fock state of $N$ electrons is $|1\cdots1\,0\cdots0\rangle$ as in the book's
figure. $h_{pqrs}$ pairs $\chi_s$ with $\chi_p$ at $\mathbf x_1$ and $\chi_r$ with
$\chi_q$ at $\mathbf x_2$ (the book's eq. for the integrals), with the operator string
$a_p^\dagger a_q^\dagger a_r a_s$ and the factor $\tfrac12$. Pauli strings are written in
book order, leftmost letter on qubit 0.

In [1]:
import numpy as np
from scipy.linalg import expm
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp, Statevector, Operator

np.set_printoptions(precision=6, suppress=True)

## Step 1: integrals from PySCF, and the Hartree–Fock sanity check

In [2]:
from pyscf import gto, scf, ao2mo, fci

mol = gto.M(atom="H 0 0 0; H 0 0 0.735", basis="sto-3g", unit="angstrom", verbose=0)
mf = scf.RHF(mol).run()
n_orb = mf.mo_coeff.shape[1]                        # spatial orbitals (2 for H2/STO-3G)
C = mf.mo_coeff
h_core = C.T @ mf.get_hcore() @ C                   # h_ij in the MO basis
eri = ao2mo.restore(1, ao2mo.kernel(mol, C), n_orb)  # (ij|kl), chemists' notation, MO basis
E_nuc = mol.energy_nuc()
E_HF_pyscf = mf.e_tot
E_FCI_pyscf = fci.FCI(mf).kernel()[0]
print(f"spatial orbitals: {n_orb}, nuclear repulsion {E_nuc:.6f}")
print(f"PySCF: E_HF = {E_HF_pyscf:.6f}, E_FCI = {E_FCI_pyscf:.6f} Hartree")

M = 2 * n_orb                                       # spin-orbitals
N_el = mol.nelectron
spin_orb = lambda p: (p // 2, p % 2)                # interleaved: p = 2 i + sigma

h1 = np.zeros((M, M))
h2 = np.zeros((M, M, M, M))
for p in range(M):
    for q in range(M):
        ip, sp = spin_orb(p); iq, sq = spin_orb(q)
        if sp == sq:
            h1[p, q] = h_core[ip, iq]
        for r in range(M):
            for s in range(M):
                ir, sr = spin_orb(r); is_, ss = spin_orb(s)
                # chi_s pairs with chi_p at x1, chi_r with chi_q at x2  ->  (ps|qr)
                if sp == ss and sq == sr:
                    h2[p, q, r, s] = eri[ip, is_, iq, ir]

# Hartree-Fock energy recomputed from h1, h2 with the occupied set {0,...,N-1}
occ = list(range(N_el))
E_HF_check = sum(h1[p, p] for p in occ) + 0.5 * sum(h2[p, q, q, p] - h2[p, q, p, q] for p in occ for q in occ) + E_nuc
print(f"recomputed E_HF = {E_HF_check:.6f}  (matches PySCF: {abs(E_HF_check - E_HF_pyscf) < 1e-8})")

spatial orbitals: 2, nuclear repulsion 0.719969
PySCF: E_HF = -1.116999, E_FCI = -1.137306 Hartree
recomputed E_HF = -1.116999  (matches PySCF: True)


## Steps 2 and 3: $\beta$, its inverse over $\mathbb{F}_2$, and the three sets

$\beta_1=(1)$, $\beta_{2m}=\begin{pmatrix}\beta_m&0\\A&\beta_m\end{pmatrix}$ with $A$ all zeros
except a bottom row of ones. $\pi$ is the lower-triangular matrix of ones.

In [3]:
def beta_matrix(M):
    B = np.array([[1]], dtype=int)
    while B.shape[0] < M:
        m = B.shape[0]
        A = np.zeros((m, m), dtype=int); A[-1, :] = 1
        B = np.block([[B, np.zeros((m, m), dtype=int)], [A, B]])
    return B[:M, :M]            # M a power of two here; the book's recursion assumes it


def inverse_mod2(B):
    n = B.shape[0]
    aug = np.concatenate([B.copy() % 2, np.eye(n, dtype=int)], axis=1)
    row = 0
    for col in range(n):
        piv = next((r for r in range(row, n) if aug[r, col] == 1), None)
        if piv is None:
            raise ValueError("singular over F2")
        aug[[row, piv]] = aug[[piv, row]]
        for r in range(n):
            if r != row and aug[r, col] == 1:
                aug[r] ^= aug[row]
        row += 1
    return aug[:, n:]


def bk_sets(M):
    B = beta_matrix(M)
    Binv = inverse_mod2(B)
    assert np.array_equal((B @ Binv) % 2, np.eye(M, dtype=int)), "beta * beta^-1 != I mod 2"
    pi = np.tril(np.ones((M, M), dtype=int))
    piBinv = (pi @ Binv) % 2
    sets = {}
    for j in range(M):
        U = [i for i in range(j + 1, M) if B[i, j]]                     # column j, rows > j
        P = [i for i in range(M) if j > 0 and piBinv[j - 1, i]]          # row j-1 of pi beta^-1
        F = [i for i in range(j) if Binv[j, i]]                         # row j of beta^-1, cols < j
        R = [i for i in P if i not in F]
        if j % 2 == 0:
            assert F == [], f"F({j}) must be empty for even j"
        sets[j] = dict(U=U, P=P, F=F, R=R, rho=P if j % 2 == 0 else R)   # even qubits store one occupation
    return B, sets


B8, sets8 = bk_sets(8)
print("M=8 sets (compare the book's table):")
for j, s in sets8.items():
    print(f"  j={j}: U={s['U']}  P={s['P']}  F={s['F']}  R={s['R']}")

M=8 sets (compare the book's table):
  j=0: U=[1, 3, 7]  P=[]  F=[]  R=[]
  j=1: U=[3, 7]  P=[0]  F=[0]  R=[]
  j=2: U=[3, 7]  P=[1]  F=[]  R=[1]
  j=3: U=[7]  P=[1, 2]  F=[1, 2]  R=[]
  j=4: U=[5, 7]  P=[3]  F=[]  R=[3]
  j=5: U=[7]  P=[3, 4]  F=[4]  R=[3]
  j=6: U=[7]  P=[3, 5]  F=[]  R=[3, 5]
  j=7: U=[]  P=[3, 5, 6]  F=[3, 5, 6]  R=[]


## Step 4: the operator mapping in the symplectic representation

$a_j^\dagger \mapsto X_{U(j)}\,\Pi_j^+\,Z_{\rho(j)}$ with $\Pi_j^\pm = A_j^\pm E_{F(j)} - A_j^\mp O_{F(j)}$,
$\rho(j)=P(j)$ for even $j$ (which stores one occupation) and $R(j)$ for odd $j$ (a partial
sum). Jordan–Wigner is the same formula with $U=\emptyset$, $F=\emptyset$,
$\rho(j)=P(j)=\{0,\dots,j-1\}$. The parity encoding ($\beta=\pi$) has
$U(j)=\{j+1,\dots,M-1\}$, $P(j)=F(j)=\{j-1\}$, and since every qubit there stores a partial
sum, $\rho(j)=R(j)=\emptyset$ for all $j$, giving $a_j^\dagger\mapsto\tfrac12X_{U(j)}(X_jZ_{j-1}-iY_j)$.

In [4]:
def pauli_on(M, letters):
    """SparsePauliOp of a book-ordered string from {qubit: letter}."""
    s = ["I"] * M
    for q, L in letters.items():
        s[q] = L
    return SparsePauliOp("".join(s))


def Z_set(M, S):
    return pauli_on(M, {i: "Z" for i in S})


def X_set(M, S):
    return pauli_on(M, {i: "X" for i in S})


def A_plus(M, j):   # (X - iY)/2 on qubit j
    return 0.5 * (pauli_on(M, {j: "X"}) - 1j * pauli_on(M, {j: "Y"}))


def A_minus(M, j):
    return 0.5 * (pauli_on(M, {j: "X"}) + 1j * pauli_on(M, {j: "Y"}))


def ladder(M, j, dagger, sets):
    U, F, rho = sets[j]["U"], sets[j]["F"], sets[j]["rho"]
    I = SparsePauliOp("I" * M)
    E_F = 0.5 * (I + Z_set(M, F)) if F else I
    O_F = 0.5 * (I - Z_set(M, F)) if F else 0 * I
    Ap, Am = A_plus(M, j), A_minus(M, j)
    Pi = (Ap @ E_F - Am @ O_F) if dagger else (Am @ E_F - Ap @ O_F)
    op = X_set(M, U) @ Pi @ Z_set(M, rho)
    return op.simplify()


def jw_sets(M):      # no qubit stores a partial sum: rho = P always
    return {j: dict(U=[], P=list(range(j)), F=[], R=list(range(j)), rho=list(range(j))) for j in range(M)}


def parity_sets(M):  # every qubit stores a partial sum: rho = R always
    return {j: dict(U=list(range(j + 1, M)), P=[j - 1] if j > 0 else [], F=[j - 1] if j > 0 else [], R=[], rho=[]) for j in range(M)}


encodings = {"Jordan-Wigner": jw_sets(M), "parity": parity_sets(M), "Bravyi-Kitaev": bk_sets(M)[1]}

## Test 1 (the one that catches almost everything): anticommutators for $M=4$

$\{a_i,a_j\}=0$, $\{a_i^\dagger,a_j^\dagger\}=0$, $\{a_i,a_j^\dagger\}=\delta_{ij}$, as dense matrices.

In [5]:
def anticomm(A, B):
    return A @ B + B @ A


for name, sets in encodings.items():
    a = [ladder(M, j, False, sets).to_matrix() for j in range(M)]
    ad = [ladder(M, j, True, sets).to_matrix() for j in range(M)]
    I = np.eye(2**M)
    for i in range(M):
        for j in range(M):
            assert np.allclose(anticomm(a[i], a[j]), 0), (name, i, j)
            assert np.allclose(anticomm(ad[i], ad[j]), 0), (name, i, j)
            assert np.allclose(anticomm(a[i], ad[j]), I if i == j else 0), (name, i, j)
    # Test 2: number operator diagonal with integer eigenvalues 0..M
    Nop = sum(ad[j] @ a[j] for j in range(M))
    assert np.allclose(Nop, np.diag(np.diag(Nop)))
    assert set(np.round(np.real(np.diag(Nop))).astype(int)) <= set(range(M + 1))
    print(f"{name}: anticommutators OK, number operator diagonal with eigenvalues {sorted(set(int(v) for v in np.round(np.real(np.diag(Nop)))))}")

Jordan-Wigner: anticommutators OK, number operator diagonal with eigenvalues [0, 1, 2, 3, 4]
parity: anticommutators OK, number operator diagonal with eigenvalues [0, 1, 2, 3, 4]
Bravyi-Kitaev: anticommutators OK, number operator diagonal with eigenvalues [0, 1, 2, 3, 4]


## Step 5: assemble the qubit Hamiltonian and collect terms

$\hat H = \sum_{pq}h_{pq}a_p^\dagger a_q + \tfrac12\sum_{pqrs}h_{pqrs}a_p^\dagger a_q^\dagger a_r a_s$,
identical strings summed, terms below $10^{-8}$ dropped.

In [6]:
def qubit_hamiltonian(h1, h2, sets, threshold=1e-8):
    M = h1.shape[0]
    a = [ladder(M, j, False, sets) for j in range(M)]
    ad = [ladder(M, j, True, sets) for j in range(M)]
    H = 0 * SparsePauliOp("I" * M)
    for p in range(M):
        for q in range(M):
            if abs(h1[p, q]) > 1e-12:
                H += h1[p, q] * (ad[p] @ a[q])
    for p in range(M):
        for q in range(M):
            for r in range(M):
                for s in range(M):
                    if abs(h2[p, q, r, s]) > 1e-12:
                        H += 0.5 * h2[p, q, r, s] * (ad[p] @ ad[q] @ a[r] @ a[s])
    H = H.simplify(atol=threshold)
    return H


H_q = {name: qubit_hamiltonian(h1, h2, sets) for name, sets in encodings.items()}
for name, H in H_q.items():
    print(f"{name}: {len(H)} Pauli terms")
print("\nJordan-Wigner terms:")
for P, c in sorted(zip(H_q["Jordan-Wigner"].paulis.to_labels(), H_q["Jordan-Wigner"].coeffs), key=lambda t: -abs(t[1])):
    print(f"  {c.real:+.6f}  {P}")

Jordan-Wigner: 15 Pauli terms
parity: 15 Pauli terms
Bravyi-Kitaev: 15 Pauli terms

Jordan-Wigner terms:
  -0.810548  IIII
  -0.225753  IIIZ
  -0.225753  IIZI
  +0.174643  IIZZ
  +0.172184  ZIII
  +0.172184  IZII
  +0.168928  ZZII
  +0.166145  ZIIZ
  +0.166145  IZZI
  +0.120913  ZIZI
  +0.120913  IZIZ
  +0.045233  YXXY
  -0.045233  YYXX
  -0.045233  XXYY
  +0.045233  XYYX


## Tests 3, 4, 5: Hermiticity, ground-state energy against FCI, encoding agreement

In [7]:
for name, H in H_q.items():
    assert np.allclose(H.coeffs.imag, 0), f"{name}: complex coefficient -> dropped conjugate or sign error"
spectra = {}
for name, H in H_q.items():
    w = np.linalg.eigvalsh(H.to_matrix())
    spectra[name] = w
    print(f"{name}: lowest electronic eigenvalue {w[0]:.6f}, total {w[0] + E_nuc:.6f} Hartree")
assert abs(spectra["Jordan-Wigner"][0] + E_nuc - E_FCI_pyscf) < 1e-8, "ground state != FCI"
for name in encodings:
    assert np.allclose(spectra[name], spectra["Jordan-Wigner"]), f"{name} spectrum differs"
print(f"\nTest 3 passed (real coefficients); Test 4 passed (E0 + E_nuc = FCI = {E_FCI_pyscf:.6f}); Test 5 passed (all three encodings share the spectrum)")

# the Hartree-Fock state |1100> has energy E_HF under Jordan-Wigner
hf = np.zeros(2**M); hf[int("1100", 2)] = 1
E = np.real(hf @ H_q["Jordan-Wigner"].to_matrix() @ hf) + E_nuc
print(f"<HF|H|HF> + E_nuc = {E:.6f} (= E_HF: {abs(E - E_HF_pyscf) < 1e-8})")

Jordan-Wigner: lowest electronic eigenvalue -1.857275, total -1.137306 Hartree
parity: lowest electronic eigenvalue -1.857275, total -1.137306 Hartree
Bravyi-Kitaev: lowest electronic eigenvalue -1.857275, total -1.137306 Hartree

Test 3 passed (real coefficients); Test 4 passed (E0 + E_nuc = FCI = -1.137306); Test 5 passed (all three encodings share the spectrum)
<HF|H|HF> + E_nuc = -1.116999 (= E_HF: True)


## The sign trap, reproduced

Pair the integral the other way (`eri[ip, ir, iq, is_]`, i.e. $\chi_r$ with $\chi_p$ at
$\mathbf{x}_1$) and the ground-state energy comes out wrong: the book quotes $-2.46$
Hartree for H$_2$ with the wrong pairing.

In [8]:
h2_wrong = np.zeros_like(h2)
for p in range(M):
    for q in range(M):
        for r in range(M):
            for s in range(M):
                ip, sp = spin_orb(p); iq, sq = spin_orb(q); ir, sr = spin_orb(r); is_, ss = spin_orb(s)
                if sp == sr and sq == ss:
                    h2_wrong[p, q, r, s] = eri[ip, ir, iq, is_]
H_wrong = qubit_hamiltonian(h1, h2_wrong, jw_sets(M))
print(f"wrong pairing: E0 + E_nuc = {np.linalg.eigvalsh(H_wrong.to_matrix())[0] + E_nuc:.4f} Hartree  (electron repulsion became attraction)")

wrong pairing: E0 + E_nuc = -6.4072 Hartree  (electron repulsion became attraction)


## Tapering to two qubits

In the parity encoding with spin-blocked ordering (all $\alpha$ then all $\beta$), qubit
$M/2-1$ holds the $\alpha$-electron parity and qubit $M-1$ the total parity. Both are
conserved, every string touches them only with $I$ or $Z$, so replace the $Z$ by its
eigenvalue and delete the qubit. H$_2$ drops from 4 qubits and 15 terms to 2 qubits.

Note on the term count. Done this way the result has five strings, $I$, $Z_0$, $Z_1$,
$Z_0Z_1$ and $X_0X_1$: the four $XXYY$-type strings of the 4-qubit Hamiltonian all land on
$X_0X_1$ once the frozen $Z$'s are replaced by their eigenvalues. The six-term form with
separate $X_0X_1$ and $Y_0Y_1$ of equal weight (O'Malley et al. 2016, and the book's
worked example) comes from a different two-qubit reduction; on the physical subspace the
two are the same operator and the spectrum below is identical.

In [9]:
# re-index spin-orbitals to blocked order: p_blocked = i + sigma * n_orb
perm = [2 * i + s for s in range(2) for i in range(n_orb)]        # blocked index -> interleaved index
h1_b = h1[np.ix_(perm, perm)]
h2_b = h2[np.ix_(perm, perm, perm, perm)]
H_par_b = qubit_hamiltonian(h1_b, h2_b, parity_sets(M))
N_alpha = N_el // 2
fixed = {M // 2 - 1: (-1) ** N_alpha, M - 1: (-1) ** N_el}       # qubit -> Z eigenvalue
keep = [q for q in range(M) if q not in fixed]

terms = {}
for P, c in zip(H_par_b.paulis.to_labels(), H_par_b.coeffs.real):
    sign = 1
    for q, z in fixed.items():
        assert P[q] in "IZ", f"string {P} touches a frozen qubit with {P[q]}"
        if P[q] == "Z":
            sign *= z
    Pt = "".join(P[q] for q in keep)
    terms[Pt] = terms.get(Pt, 0) + sign * c
H_tap = SparsePauliOp.from_list([(P, c) for P, c in terms.items() if abs(c) > 1e-8])
print("tapered Hamiltonian:")
for P, c in zip(H_tap.paulis.to_labels(), H_tap.coeffs.real):
    print(f"  {c:+.6f}  {P}")
E0_tap = np.linalg.eigvalsh(H_tap.to_matrix())[0]
print(f"ground state: {E0_tap + E_nuc:.6f} Hartree (FCI {E_FCI_pyscf:.6f})")
assert abs(E0_tap + E_nuc - E_FCI_pyscf) < 1e-8

tapered Hamiltonian:
  -1.052373  II
  +0.397937  ZI
  -0.397937  IZ
  -0.011280  ZZ
  +0.180931  XX
ground state: -1.137306 Hartree (FCI -1.137306)


## Step 6: circuits

The reference state: $X$ on each occupied orbital under Jordan–Wigner; under
Bravyi–Kitaev, $X$ on qubit $j$ iff $b_j=\sum_i\beta_{ji}k_i \bmod 2 = 1$. The evolution:
`pauli_evolution` for each term, grouped so that commuting terms sit together, and
checked against `expm` on the tapered Hamiltonian.

In [10]:
bq = lambda j, tot: tot - 1 - j


def pauli_evolution(circuit, pauli_string, angle, qubits):
    active = [k for k, p in enumerate(pauli_string) if p != "I"]
    if not active:
        return
    for k in active:
        if pauli_string[k] == "X": circuit.h(qubits[k])
        elif pauli_string[k] == "Y": circuit.sdg(qubits[k]); circuit.h(qubits[k])
    last = qubits[active[-1]]
    for k in active[:-1]: circuit.cx(qubits[k], last)
    circuit.rz(2 * angle, last)
    for k in reversed(active[:-1]): circuit.cx(qubits[k], last)
    for k in active:
        if pauli_string[k] == "X": circuit.h(qubits[k])
        elif pauli_string[k] == "Y": circuit.h(qubits[k]); circuit.s(qubits[k])


# reference states
k_occ = np.array([1] * N_el + [0] * (M - N_el))
qc_jw = QuantumCircuit(M)
for j in range(M):
    if k_occ[j]: qc_jw.x(bq(j, M))
b_bk = (beta_matrix(M) @ k_occ) % 2
qc_bk = QuantumCircuit(M)
for j in range(M):
    if b_bk[j]: qc_bk.x(bq(j, M))
print("JW reference flips qubits", [j for j in range(M) if k_occ[j]], "; BK reference flips qubits", [j for j in range(M) if b_bk[j]])
for name, qc in [("Jordan-Wigner", qc_jw), ("Bravyi-Kitaev", qc_bk)]:
    E = Statevector(qc).expectation_value(H_q[name]).real + E_nuc
    print(f"  <ref|H|ref> under {name}: {E:.6f} (= E_HF: {abs(E - E_HF_pyscf) < 1e-8})")

# evolution under the tapered 2-qubit Hamiltonian: the Z-type strings form one commuting
# group and XX the other, so a symmetric split has two groups and the usual slopes apply
labels = H_tap.paulis.to_labels(); coeffs = H_tap.coeffs.real
groupZ = [(P, c) for P, c in zip(labels, coeffs) if set(P) <= {"I", "Z"}]
groupXY = [(P, c) for P, c in zip(labels, coeffs) if not set(P) <= {"I", "Z"}]
q2 = [bq(0, 2), bq(1, 2)]


def trotter(t, r):
    qc = QuantumCircuit(2); dt = t / r
    for _ in range(r):
        for P, c in groupZ: pauli_evolution(qc, P, c * dt / 2, q2)
        for P, c in groupXY: pauli_evolution(qc, P, c * dt, q2)
        for P, c in groupZ: pauli_evolution(qc, P, c * dt / 2, q2)
    return Operator(qc).data


# pauli_evolution skips the identity term (a global phase), so compare against the
# Hamiltonian without it; the constant is added back classically when reporting energies
H_no_id = SparsePauliOp.from_list([(P, c) for P, c in zip(labels, coeffs) if set(P) != {"I"}])
t = 1.0
exact = expm(-1j * H_no_id.to_matrix() * t)
for r in [1, 2, 4, 8]:
    print(f"r={r}: ||U_trotter - expm|| = {np.linalg.norm(trotter(t, r) - exact, 2):.2e}")

JW reference flips qubits [0, 1] ; BK reference flips qubits [0]
  <ref|H|ref> under Jordan-Wigner: -1.116999 (= E_HF: True)
  <ref|H|ref> under Bravyi-Kitaev: -1.116999 (= E_HF: True)
r=1: ||U_trotter - expm|| = 2.02e-02
r=2: ||U_trotter - expm|| = 4.79e-03
r=4: ||U_trotter - expm|| = 1.18e-03
r=8: ||U_trotter - expm|| = 2.94e-04
